# Composite DNA Dataset Generator — 4-Base Eta-Based Alphabet

**New alphabet (per collaborator request).** Each symbol is a 4-base mixture — every symbol assigns a non-zero weight to all four bases A, C, G, T. The weights live on a grid centered at 1/4 = 0.25 with step `ETA`, with a minimum weight of 0.15. With `ETA = 0.1` the grid is $\{0.15,\, 0.25,\, 0.35,\, 0.45,\, 0.55\}$:

- **35 valid 4-tuples** $(p_A, p_C, p_G, p_T)$ on the grid summing to 1
- Plus the **4 pure bases** A, C, G, T (indices 0-3), included by default
- **39 symbols total**; theoretical capacity $\log_2 39 \approx 5.29$ bits/position

Examples (all valid members of this alphabet):
- `A C G T = 0.25 0.25 0.25 0.25`  (the fully-balanced 4-mix symbol)
- `A C G T = 0.35 0.35 0.15 0.15`
- `A` $=(1,0,0,0)$  (a pure base)

By default the alphabet **includes the 4 pure bases A, C, G, T as symbols 0-3** (→ **39 symbols**), for consistency with the eta-2base (34-symbol) and eta-3mix (80-symbol) alphabets, which both include the pure bases. The pure bases sit outside the 4-mix grid (a pure base has 0-weight components, violating the 0.15 minimum), so they are added explicitly. Set `INCLUDE_PURE_BASES = False` for the pure-4-mix-only variant (35 symbols, $\log_2 35 \approx 5.13$ bits/position).

**Recommended single-point configuration**: `M = 25, profile = EZ17` — same as the 3-mix study, for a clean comparison. Errors are injected via the same two-stage SDG channel as the prior dataset notebooks.

In [1]:
# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import os
import time
from collections import Counter
from datetime import datetime

In [2]:
# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

# ------------------- SELECT ERROR MODEL(S) -------------------
# Single profile, list, or "ALL" (all 8 profiles).
# Recommended for this study: "EZ17"  (or ["EZ17","R21"]).
ERROR_MODEL = "EZ17"

# ------------------- ETA-4MIX ALPHABET PARAMETERS -------------------
ETA                = 0.1
CENTER             = 0.25   # 1/4 (all four bases active)
MIN_WEIGHT         = 0.15   # smallest allowed per-base weight
INCLUDE_PURE_BASES = True   # True -> also include A,C,G,T pure bases as symbols 0-3 (39 symbols);
                            # False -> pure 4-mix only (35 symbols).
                            # Default True for consistency with the eta-2base and eta-3mix alphabets.

# ------------------- GLOBAL DATASET PARAMETERS -------------------
NUM_SAMPLES  = 100000
MAX_COVERAGE = 50
SEED         = 42
DATASET_DIR  = "./dataset"


ERROR_MODEL_SPECS = {
    "EZ17":  {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "G15":   {"full_length": 117, "index_length": 13, "seq_length": 104,
              "name": "G15",   "platform": "Illumina miSeq + CustomArray",
              "best_eps_klml": 0.05},
    "O17":   {"full_length": 110, "index_length": 33, "seq_length": 77,
              "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "R21":   {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "R21",   "platform": "Oxford Nanopore MinION + Twist Bioscience",
              "best_eps_klml": 0.1},
    "B22":   {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "B22",   "platform": "Nanopore MinION short-read + Twist Bioscience",
              "best_eps_klml": 0.1},
    "BOS22": {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "BOS22", "platform": "Illumina miSeq 2022 + Twist Bioscience",
              "best_eps_klml": 0.001},
    "NP22":  {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "NP22",  "platform": "Nanopore Pilot Nov-2022 + Twist Bioscience",
              "best_eps_klml": 0.1},
    "NPF22": {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "NPF22", "platform": "Nanopore Full Pool Nov-2022 + Twist Bioscience",
              "best_eps_klml": 0.1},
}
ERROR_MODEL_ALIASES = {"erlich": "EZ17", "grass": "G15", "organick": "O17"}
def resolve_error_model(name):
    return ERROR_MODEL_ALIASES.get(name, name)

ALL_PROFILES = ["EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"]

def _resolve_to_list(spec):
    if isinstance(spec, str):
        if spec.upper() == "ALL":
            return list(ALL_PROFILES)
        return [resolve_error_model(spec)]
    elif isinstance(spec, (list, tuple)):
        return [resolve_error_model(s) for s in spec]
    raise TypeError(f"ERROR_MODEL must be str or list, got {type(spec)}")

PROFILES_TO_RUN = _resolve_to_list(ERROR_MODEL)
for p in PROFILES_TO_RUN:
    assert p in ERROR_MODEL_SPECS, f"Unknown profile {p}"

os.makedirs(DATASET_DIR, exist_ok=True)

def make_config_for_profile(profile_name):
    specs = ERROR_MODEL_SPECS[profile_name]
    pure_tag = "withpure" if INCLUDE_PURE_BASES else "nopure"
    alphabet_mode = f"eta4mix_{ETA}_{pure_tag}"
    cfg = {
        "error_model": profile_name,
        "error_specs": specs,
        "eta": ETA,
        "center": CENTER,
        "min_weight": MIN_WEIGHT,
        "include_pure_bases": INCLUDE_PURE_BASES,
        "alphabet_mode": alphabet_mode,
        "num_samples": NUM_SAMPLES,
        "seq_length": specs["seq_length"],
        "max_coverage": MAX_COVERAGE,
        "dataset_dir": DATASET_DIR,
        "seed": SEED,
    }
    name = f"dna_{specs['name']}_{alphabet_mode}"
    cfg["dataset_path"] = f"{DATASET_DIR}/{name}_{NUM_SAMPLES}_{MAX_COVERAGE}.pkl"
    return cfg

print("=" * 72)
print("DATASET GENERATION CONFIGURATION  (4-base eta alphabet)")
print("=" * 72)
print(f"   Profiles        : {', '.join(PROFILES_TO_RUN)}")
print(f"   Eta             : {ETA}")
print(f"   Center          : {CENTER}  (all four bases active)")
print(f"   Min weight      : {MIN_WEIGHT}")
print(f"   Include pure    : {INCLUDE_PURE_BASES}")
print(f"   Num samples     : {NUM_SAMPLES:,}")
print(f"   Max coverage    : {MAX_COVERAGE}")
print(f"   Seed            : {SEED}")
print(f"   Output dir      : {DATASET_DIR}/")
print()
for p in PROFILES_TO_RUN:
    cfg = make_config_for_profile(p)
    print(f"     • {p:<6} (n={cfg['seq_length']})  →  {cfg['dataset_path']}")
print("=" * 72)

DATASET GENERATION CONFIGURATION  (4-base eta alphabet)
   Profiles        : EZ17
   Eta             : 0.1
   Center          : 0.25  (all four bases active)
   Min weight      : 0.15
   Include pure    : True
   Num samples     : 100,000
   Max coverage    : 50
   Seed            : 42
   Output dir      : ./dataset/

     • EZ17   (n=136)  →  ./dataset/dna_EZ17_eta4mix_0.1_withpure_100000_50.pkl


In [3]:
# =============================================================================
# CELL 3: SEED HELPER
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

print(f'Seed helper ready. Base seed = {SEED}.')

Seed helper ready. Base seed = 42.


In [4]:
# =============================================================================
# CELL 4: BUILD 4-BASE ETA ALPHABET
# =============================================================================
try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False

def build_eta_4mix_alphabet(eta, center=0.25, min_weight=0.15, include_pure_bases=False):
    """Build the 4-base mixture alphabet centered at `center` (=1/4) with step `eta`.

    Every symbol assigns a non-zero weight to ALL FOUR bases A, C, G, T. The
    weights live on the grid
        {center + l*eta : center + l*eta >= min_weight, and <= the max forced by sum=1}
    With center=0.25, eta=0.1, min_weight=0.15 the grid is
        {0.15, 0.25, 0.35, 0.45, 0.55}
    and we keep every ordered 4-tuple (p_A, p_C, p_G, p_T) on that grid whose
    entries sum to exactly 1. That yields 35 four-base mixture symbols.

    log2(35) ≈ 5.13 bits/position for the mixtures-only variant.

    If include_pure_bases is True (the DEFAULT for this study), the 4 pure
    bases (A,C,G,T) are prepended as symbols at indices 0..3, giving 39
    symbols total, log2(39) ≈ 5.29 bits/position. This matches the eta-2base
    and eta-3mix alphabets, which both include the pure bases. The pure bases
    have a base with weight 0 (which lies outside the min_weight grid), so
    they are added explicitly rather than enumerated.

    Returns
    -------
    composite_map: dict[str, list[(base, prob)]]
        symbol name -> component (base, probability) list (always 4 entries for
        mixtures; 1 entry for pure bases when included).
    symbol_to_idx: dict[str, int]
    ideal_vectors: torch.Tensor of shape (V, 4)   (rows are [A,C,G,T] vectors)
    """
    import itertools
    bases = ['A', 'C', 'G', 'T']

    # Build the grid. Lower bound = min_weight. Upper bound: a base can be at
    # most 1 - 3*min_weight (when the other three are at the minimum).
    max_weight = 1.0 - 3.0 * min_weight
    # Enumerate integer offsets l such that center + l*eta is within [min, max].
    grid = []
    l = 0
    # search downward
    while center + (l - 1) * eta >= min_weight - 1e-9:
        l -= 1
    l_lo = l
    l = 0
    while center + (l + 1) * eta <= max_weight + 1e-9:
        l += 1
    l_hi = l
    grid = [center + ll * eta for ll in range(l_lo, l_hi + 1)]

    composite_map = {}
    symbol_to_idx = {}
    ideal_vectors_list = []
    cur = 0

    # Optionally include pure bases first (indices 0..3).
    if include_pure_bases:
        for bi, b in enumerate(bases):
            composite_map[b] = [(b, 1.0)]
            symbol_to_idx[b] = cur
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[bi] = 1.0
            ideal_vectors_list.append(vec)
            cur += 1

    # Enumerate 4-mix symbols: ordered grid 4-tuples summing to 1.
    for quad in itertools.product(grid, repeat=4):
        if abs(sum(quad) - 1.0) < 1e-9:
            # Name encodes the integer offsets (p - center)/eta for A,C,G,T.
            ells = [int(round((p - center) / eta)) for p in quad]
            name = "Q_" + "_".join(
                (f"{e:+d}" if e != 0 else "0") for e in ells
            )
            comps = list(zip(bases, quad))
            composite_map[name] = comps
            symbol_to_idx[name] = cur
            ideal_vectors_list.append(list(quad))
            cur += 1

    import torch
    ideal_vectors = torch.tensor(ideal_vectors_list, dtype=torch.float32)
    return composite_map, symbol_to_idx, ideal_vectors

COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS = build_eta_4mix_alphabet(
    ETA, CENTER, MIN_WEIGHT, INCLUDE_PURE_BASES)
ALL_SYMBOLS = list(COMPOSITE_MAP.keys())
VOCAB_SIZE  = len(SYMBOL_TO_IDX)

print(f"4-base eta alphabet (eta={ETA}, center={CENTER}, min_weight={MIN_WEIGHT}):")
print(f"   Include pure bases : {INCLUDE_PURE_BASES}")
print(f"   Total symbols      : {VOCAB_SIZE}")
print(f"   Capacity per pos.  : {np.log2(VOCAB_SIZE):.4f} bits")
print()
print("First 12 symbols:")
print(f"   {'idx':>4}  {'symbol':<18}  ideal vector [A, C, G, T]")
for sym, idx in sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1])[:12]:
    v = IDEAL_VECTORS[idx].tolist() if HAVE_TORCH else IDEAL_VECTORS[idx]
    v_str = '[' + ', '.join(f'{x:.4f}' for x in v) + ']'
    print(f"   {idx:>4}  {sym:<18}  {v_str}")

4-base eta alphabet (eta=0.1, center=0.25, min_weight=0.15):
   Include pure bases : True
   Total symbols      : 39
   Capacity per pos.  : 5.2854 bits

First 12 symbols:
    idx  symbol              ideal vector [A, C, G, T]
      0  A                   [1.0000, 0.0000, 0.0000, 0.0000]
      1  C                   [0.0000, 1.0000, 0.0000, 0.0000]
      2  G                   [0.0000, 0.0000, 1.0000, 0.0000]
      3  T                   [0.0000, 0.0000, 0.0000, 1.0000]
      4  Q_-1_-1_-1_+3       [0.1500, 0.1500, 0.1500, 0.5500]
      5  Q_-1_-1_0_+2        [0.1500, 0.1500, 0.2500, 0.4500]
      6  Q_-1_-1_+1_+1       [0.1500, 0.1500, 0.3500, 0.3500]
      7  Q_-1_-1_+2_0        [0.1500, 0.1500, 0.4500, 0.2500]
      8  Q_-1_-1_+3_-1       [0.1500, 0.1500, 0.5500, 0.1500]
      9  Q_-1_0_-1_+2        [0.1500, 0.2500, 0.1500, 0.4500]
     10  Q_-1_0_0_+1         [0.1500, 0.2500, 0.2500, 0.3500]
     11  Q_-1_0_+1_0         [0.1500, 0.2500, 0.3500, 0.2500]


In [5]:
# =============================================================================
# CELL 5: ERROR-RATES CLASS (8 SDG profiles, verbatim from github)
# =============================================================================

class ErrorRates:
    """8 SDG profiles. Per-base values transcribed verbatim from
    cluster_generator.py + error_rates_setup.py of the Bar-Lev SDG."""

    LONG_DELETION_LENGTH_RATES = {
        2: 2.8e-4, 3: 7.75e-5, 4: 3.25e-5, 5: 1.0e-6, 6: 5.5e-8,
    }

    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
        }
        self.profile_name = None
        self.long_deletion_length_rates = dict(self.LONG_DELETION_LENGTH_RATES)

    def set_EZ17_values(self):
        self.profile_name = "EZ17"
        self.general_errors = {'s': 1.32e-03, 'i': 5.81e-04, 'd': 9.58e-04, 'ld': 2.33e-04}
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'pi': 0.00059, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'pi': 0.00058, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'pi': 0.00057, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'pi': 0.00058, 'd': 0.00096, 'ld': 0.00023}

    def set_G15_values(self):
        self.profile_name = "G15"
        self.general_errors = {'s': 5.84e-03, 'i': 8.57e-04, 'd': 5.37e-03, 'ld': 3.48e-04}
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.00090, 'pi': 0.00092, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'pi': 0.00081, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'pi': 0.00087, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'pi': 0.00084, 'd': 0.00559, 'ld': 0.00036}

    def set_O17_values(self):
        self.profile_name = "O17"
        self.general_errors = {'s': 2.52e-03, 'i': 4.14e-04, 'd': 6.94e-04, 'ld': 2.11e-04}
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.00030, 'pi': 0.00120, 'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'pi': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'pi': 0.00029, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'pi': 0.00009, 'd': 0.00014, 'ld': 0.00006}

    def set_R21_values(self):
        self.profile_name = "R21"
        self.general_errors = {'s': 1.08e-02, 'i': 1.65e-02, 'd': 1.18e-02, 'ld': 3.36e-03}
        self.per_base_errors['A'] = {'s': 1.039e-02, 'i': 1.610e-02, 'pi': 1.585e-02, 'd': 1.192e-02, 'ld': 0.314e-02}
        self.per_base_errors['C'] = {'s': 1.042e-02, 'i': 1.639e-02, 'pi': 1.578e-02, 'd': 1.260e-02, 'ld': 0.334e-02}
        self.per_base_errors['G'] = {'s': 1.094e-02, 'i': 1.604e-02, 'pi': 1.700e-02, 'd': 1.267e-02, 'ld': 0.337e-02}
        self.per_base_errors['T'] = {'s': 1.131e-02, 'i': 1.738e-02, 'pi': 1.729e-02, 'd': 1.327e-02, 'ld': 0.357e-02}

    def set_B22_values(self):
        self.profile_name = "B22"
        self.general_errors = {'s': 1.12e-02, 'i': 1.08e-02, 'd': 7.87e-03, 'ld': 2.05e-03}
        self.per_base_errors['A'] = {'s': 0.01146, 'i': 0.01104, 'pi': 0.01096, 'd': 0.00798, 'ld': 0.00210}
        self.per_base_errors['C'] = {'s': 0.01117, 'i': 0.01092, 'pi': 0.01088, 'd': 0.00804, 'ld': 0.00208}
        self.per_base_errors['G'] = {'s': 0.01129, 'i': 0.01073, 'pi': 0.01071, 'd': 0.00792, 'ld': 0.00201}
        self.per_base_errors['T'] = {'s': 0.01094, 'i': 0.01039, 'pi': 0.01053, 'd': 0.00769, 'ld': 0.00201}

    def set_BOS22_values(self):
        self.profile_name = "BOS22"
        self.general_errors = {'s': 5.29e-04, 'i': 5.42e-05, 'd': 7.08e-05, 'ld': 1.81e-05}
        self.per_base_errors['A'] = {'s': 0.004e-2, 'i': 0.004e-2, 'pi': 0.0,      'd': 0.0,      'ld': 0.0}
        self.per_base_errors['C'] = {'s': 0.024e-2, 'i': 0.006e-2, 'pi': 0.002e-2, 'd': 0.003e-2, 'ld': 0.001e-2}
        self.per_base_errors['G'] = {'s': 0.004e-2, 'i': 0.005e-2, 'pi': 0.0,      'd': 0.0,      'ld': 0.0}
        self.per_base_errors['T'] = {'s': 0.175e-2, 'i': 0.007e-2, 'pi': 0.018e-2, 'd': 0.024e-2, 'ld': 0.006e-2}

    def set_NP22_values(self):
        self.profile_name = "NP22"
        self.general_errors = {'s': 1.29e-02, 'i': 1.16e-02, 'd': 9.75e-03, 'ld': 2.82e-03}
        self.per_base_errors['A'] = {'s': 0.520e-2, 'i': 0.644e-2, 'pi': 0.582e-2, 'd': 0.373e-2, 'ld': 0.098e-2}
        self.per_base_errors['C'] = {'s': 1.498e-2, 'i': 1.331e-2, 'pi': 1.340e-2, 'd': 1.178e-2, 'ld': 0.336e-2}
        self.per_base_errors['G'] = {'s': 0.638e-2, 'i': 0.735e-2, 'pi': 0.667e-2, 'd': 0.474e-2, 'ld': 0.128e-2}
        self.per_base_errors['T'] = {'s': 2.437e-2, 'i': 1.867e-2, 'pi': 1.983e-2, 'd': 1.903e-2, 'ld': 0.550e-2}

    def set_NPF22_values(self):
        self.profile_name = "NPF22"
        self.general_errors = {'s': 1.56e-02, 'i': 1.24e-02, 'd': 9.79e-03, 'ld': 2.67e-03}
        self.per_base_errors['A'] = {'s': 1.700e-2, 'i': 1.320e-2, 'pi': 1.343e-2, 'd': 1.095e-2, 'ld': 0.294e-2}
        self.per_base_errors['C'] = {'s': 1.606e-2, 'i': 1.271e-2, 'pi': 1.272e-2, 'd': 1.038e-2, 'ld': 0.277e-2}
        self.per_base_errors['G'] = {'s': 1.593e-2, 'i': 1.256e-2, 'pi': 1.255e-2, 'd': 1.014e-2, 'ld': 0.273e-2}
        self.per_base_errors['T'] = {'s': 1.358e-2, 'i': 1.116e-2, 'pi': 1.094e-2, 'd': 0.849e-2, 'ld': 0.224e-2}

    def set_values_by_model(self, model_name):
        dispatch = {
            "erlich":   self.set_EZ17_values,  "EZ17":  self.set_EZ17_values,
            "grass":    self.set_G15_values,   "G15":   self.set_G15_values,
            "organick": self.set_O17_values,   "O17":   self.set_O17_values,
            "R21":      self.set_R21_values,
            "B22":      self.set_B22_values,
            "BOS22":    self.set_BOS22_values,
            "NP22":     self.set_NP22_values,
            "NPF22":    self.set_NPF22_values,
        }
        if model_name not in dispatch:
            raise ValueError(f"Unknown error model: {model_name}. Valid: {sorted(dispatch.keys())}")
        dispatch[model_name]()

    def print_current_values(self):
        print(f"   Profile: {self.profile_name}")
        g = self.general_errors
        print(f"   General: s={g['s']:.5e}  i={g['i']:.5e}  d={g['d']:.5e}  ld={g['ld']:.5e}")
        for b in 'ACGT':
            r = self.per_base_errors[b]
            print(f"   {b}: s={r['s']:.5e}  pi(ins)={r['pi']:.5e}  d={r['d']:.5e}  ld={r['ld']:.5e}  i(weight)={r['i']:.5e}")


In [6]:
# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS
# =============================================================================
def generate_composite_sequence(length, composite_map):
    """Sample a uniform random composite-symbol sequence of given length."""
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]

def realize_sequence(composite_seq, composite_map):
    """Realize each composite symbol into a single concrete nucleotide.
    For 4-base mixtures, draw from the 4 components with the symbol's weights."""
    out = []
    for sym in composite_seq:
        comps = composite_map[sym]
        if len(comps) == 1:
            out.append(comps[0][0])
        else:
            bases = [c[0] for c in comps]
            probs = [c[1] for c in comps]
            out.append(random.choices(bases, weights=probs, k=1)[0])
    return ''.join(out)

def apply_ids_noise(sequence, error_profile):
    """Two-stage IDS+long-deletion noise injector (matches Bar-Lev SDG)."""
    bases = ['A', 'C', 'G', 'T']
    g = error_profile.general_errors
    p_total_err = g['s'] + g['i'] + g['d'] + g['ld']
    if p_total_err > 1.0:
        p_total_err = 1.0
    ld_lengths = list(error_profile.long_deletion_length_rates.keys())
    ld_weights = list(error_profile.long_deletion_length_rates.values())
    ins_base_weights = [error_profile.per_base_errors[x]['i'] for x in bases]
    if sum(ins_base_weights) <= 0:
        ins_base_weights = [1.0, 1.0, 1.0, 1.0]

    out_chars = []
    n = len(sequence)
    pos = 0
    while pos < n:
        b = sequence[pos]
        if b not in bases:
            out_chars.append(b)
            pos += 1
            continue
        if random.random() >= p_total_err:
            out_chars.append(b)
            pos += 1
            continue
        pbr = error_profile.per_base_errors[b]
        event_types   = ['s', 'pi', 'd', 'ld']
        event_weights = [pbr['s'], pbr['pi'], pbr['d'], pbr['ld']]
        if sum(event_weights) <= 0:
            out_chars.append(b); pos += 1; continue
        event = random.choices(event_types, weights=event_weights, k=1)[0]
        if event == 's':
            opts = [x for x in bases if x != b]
            out_chars.append(random.choice(opts))
            pos += 1
        elif event == 'pi':
            # insertion: emit inserted base then the original base, advance
            # past it (original base consumed -- matches github index logic)
            ins_b = random.choices(bases, weights=ins_base_weights, k=1)[0]
            out_chars.append(ins_b)
            out_chars.append(b)
            pos += 1
        elif event == 'd':
            pos += 1
        elif event == 'ld':
            L = random.choices(ld_lengths, weights=ld_weights, k=1)[0]
            pos += L
    return "".join(out_chars)


In [7]:
# =============================================================================
# CELL 7: MAIN DATASET-GENERATION FUNCTION
# =============================================================================
def generate_dataset_4mix(config, composite_map, symbol_to_idx, ideal_vectors):
    errors = ErrorRates()
    errors.set_values_by_model(config['error_model'])
    errors.print_current_values()

    n_samples = config['num_samples']
    seq_len   = config['seq_length']
    cov       = config['max_coverage']
    fname     = config['dataset_path']

    ideal_list = ideal_vectors.tolist() if HAVE_TORCH else ideal_vectors
    dataset = {
        'metadata': {
            'type': f"Composite DNA 4-base Eta (eta={config['eta']})",
            'error_profile': config['error_specs']['name'],
            'platform': config['error_specs']['platform'],
            'num_samples': n_samples,
            'seq_length': seq_len,
            'coverage_depth': cov,
            'vocab_size': len(symbol_to_idx),
            'eta': config['eta'],
            'center': config['center'],
            'min_weight': config['min_weight'],
            'include_pure_bases': config['include_pure_bases'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_list,
            'composite_map': {k: list(v) for k, v in composite_map.items()},
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seed': config['seed'],
        },
        'data': [],
    }

    print(f"\n{'='*64}")
    print(' GENERATING 4-BASE ETA DATASET')
    print(f"{'='*64}")
    print(f"   Samples       : {n_samples:,}")
    print(f"   Seq length    : {seq_len}")
    print(f"   Coverage      : {cov}")
    print(f"   Vocab         : {len(symbol_to_idx)} symbols")
    print(f"{'='*64}")

    t0 = time.time()
    for i in range(n_samples):
        clean = generate_composite_sequence(seq_len, composite_map)
        reads = []
        for _ in range(cov):
            realized = realize_sequence(clean, composite_map)
            reads.append(apply_ids_noise(realized, errors))
        dataset['data'].append({'id': i, 'label': clean, 'cluster': reads})
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta_s = (n_samples - i - 1) / rate
            print(f'   Processed {i+1:,}/{n_samples:,} | {rate:.1f} samples/s | ETA {eta_s:.1f}s')

    with open(fname, 'wb') as f:
        pickle.dump(dataset, f)
    total = time.time() - t0
    print(f"\n   Output file : {fname}")
    print(f"   Total time  : {total:.1f}s  ({total/60:.1f} min)")
    print(f"   File size   : {os.path.getsize(fname) / (1024*1024):.1f} MB")
    return dataset

In [8]:
# =============================================================================
# CELL 8: ANALYZE DATASET
# =============================================================================
def analyze_dataset(dataset, symbol_to_idx):
    all_syms = [s for sample in dataset['data'] for s in sample['label']]
    counter = Counter(all_syms)
    total = len(all_syms)
    expected = 100.0 / len(symbol_to_idx)
    print(f"\n{'='*64}")
    print(' DATASET STATISTICS')
    print(f"{'='*64}")
    print(f"\nSymbol distribution (first 15 of {len(symbol_to_idx)}):")
    print(f"   {'Symbol':<18} {'Count':>10} {'%':>10} {'Expected':>10}")
    for sym in sorted(counter.keys(), key=lambda x: symbol_to_idx[x])[:15]:
        c = counter[sym]
        print(f"   {sym:<18} {c:>10,} {100*c/total:>9.2f}% {expected:>9.2f}%")
    if len(symbol_to_idx) > 15:
        print(f"   ... ({len(symbol_to_idx)-15} more symbols)")
    print(f"   Total emitted symbols: {total:,}")

    read_lens = [len(r) for s in dataset['data'] for r in s['cluster']]
    print(f"\nRead length statistics:")
    print(f"   Original length    : {dataset['metadata']['seq_length']}")
    print(f"   Mean read length   : {np.mean(read_lens):.2f}")
    print(f"   Std read length    : {np.std(read_lens):.2f}")
    print(f"   Min / Max          : {np.min(read_lens)} / {np.max(read_lens)}")

In [9]:
# =============================================================================
# CELL 9: MAIN EXECUTION (single profile or batch)
# =============================================================================
OVERWRITE = False
results_summary = []
t_batch_start = time.time()

for run_idx, profile_name in enumerate(PROFILES_TO_RUN, start=1):
    cfg = make_config_for_profile(profile_name)

    print(f"\n\n{'#'*72}")
    print(f"#  [{run_idx}/{len(PROFILES_TO_RUN)}]  Profile: {profile_name}  "
          f"({cfg['error_specs']['platform']})")
    print(f"#  n={cfg['seq_length']}  eta={ETA}  vocab={VOCAB_SIZE}  "
          f"N={NUM_SAMPLES:,}  M={MAX_COVERAGE}")
    print(f"{'#'*72}")

    if (not OVERWRITE) and os.path.exists(cfg['dataset_path']):
        size_mb = os.path.getsize(cfg['dataset_path']) / (1024*1024)
        print(f"[skip] Dataset already exists ({size_mb:.1f} MB): {cfg['dataset_path']}")
        print('       Set OVERWRITE = True to force regeneration.')
        results_summary.append({'profile': profile_name, 'status': 'skipped (exists)',
                                'path': cfg['dataset_path'], 'time_s': 0.0})
        continue

    set_seed(SEED)
    t0 = time.time()
    try:
        dataset = generate_dataset_4mix(cfg, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
        analyze_dataset(dataset, SYMBOL_TO_IDX)
        dt = time.time() - t0
        results_summary.append({'profile': profile_name, 'status': 'ok',
                                'path': cfg['dataset_path'], 'time_s': dt})
    except Exception as e:
        dt = time.time() - t0
        print(f'\n[ERROR] Profile {profile_name} failed: {type(e).__name__}: {e}')
        results_summary.append({'profile': profile_name, 'status': f'FAILED: {e}',
                                'path': cfg['dataset_path'], 'time_s': dt})
        continue

t_batch = time.time() - t_batch_start
print(f"\n\n{'='*72}\n BATCH SUMMARY\n{'='*72}")
print(f"   {'Profile':<8} {'Status':<22} {'Time':>10}   File")
print(f"   {'-'*8} {'-'*22} {'-'*10}   {'-'*40}")
for r in results_summary:
    t_s = f"{r['time_s']:.1f}s" if r['time_s'] > 0 else '—'
    print(f"   {r['profile']:<8} {r['status']:<22} {t_s:>10}   {r['path']}")
print(f"   {'-'*72}")
n_ok = sum(1 for r in results_summary if r['status'] == 'ok')
n_skip = sum(1 for r in results_summary if 'skipped' in r['status'])
n_fail = sum(1 for r in results_summary if 'FAILED' in r['status'])
print(f"   {n_ok} generated, {n_skip} skipped, {n_fail} failed.")
print(f"   Total wall time: {t_batch:.1f}s  ({t_batch/60:.1f} min)\n{'='*72}")



########################################################################
#  [1/1]  Profile: EZ17  (Illumina miSeq + Twist Bioscience)
#  n=136  eta=0.1  vocab=39  N=100,000  M=50
########################################################################
   Profile: EZ17
   General: s=1.32000e-03  i=5.81000e-04  d=9.58000e-04  ld=2.33000e-04
   A: s=1.35000e-03  pi(ins)=5.90000e-04  d=9.90000e-04  ld=2.40000e-04  i(weight)=5.70000e-04
   C: s=1.35000e-03  pi(ins)=5.80000e-04  d=9.80000e-04  ld=2.30000e-04  i(weight)=5.90000e-04
   G: s=1.26000e-03  pi(ins)=5.70000e-04  d=9.40000e-04  ld=2.30000e-04  i(weight)=5.90000e-04
   T: s=1.32000e-03  pi(ins)=5.80000e-04  d=9.60000e-04  ld=2.30000e-04  i(weight)=5.80000e-04

 GENERATING 4-BASE ETA DATASET
   Samples       : 100,000
   Seq length    : 136
   Coverage      : 50
   Vocab         : 39 symbols
   Processed 10,000/100,000 | 66.2 samples/s | ETA 1358.6s
   Processed 20,000/100,000 | 66.5 samples/s | ETA 1203.8s
   Processed 30,000/100,0